In [1]:
!pip install --upgrade openai

In [ ]:
"""
Hotel Image Selection AI Pipeline
========================================
本程式為自動化 Hotel 圖片篩選流程，透過串接 OpenAI GPT-4o Vision 模型，
進行圖片品質過濾、分類、精選與最終排序的自動化流程。

處理階段:
----------------------------------------
Stage 0: 抓取圖片資訊
Stage 1: 圖片品質過濾、分類與類別校正
Stage 2: 各類別內部圖片精選
Stage 3: 依據填充策略進行最終排序與輸出

"""

# ==========================================================
# IMPORTS
# ==========================================================

import requests
import asyncio
import json
import os
import csv
import random
from collections import defaultdict
from openai import AsyncOpenAI, RateLimitError
from google.colab import userdata, files
from google.colab import drive


# ==========================================================
# SETTINGS
# 系統設定
# ==========================================================

drive.mount('/content/drive')
os.chdir('/content/drive/My Drive/Colab Notebooks')

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HOTEL_LIST = [545854,32198776]

STAGE1_MODEL = "gpt-4o"
STAGE2_MODEL = "gpt-4o"

# Stage1 每批圖片數
BATCH_SIZE = 8

# 單一 Hotel 內 OpenAI 併發數
CONCURRENCY = 2

# Stage2 最大 Vision 輸入數
MAX_VISION_INPUT = 35

# Stage2 各類別最大保留數
CATEGORY_LIMIT = {
  "建築物外觀": 1,
  "戶外環境": 5,
  "室內公共設施": 4,
  "客房": 7,
  "餐飲": 3
}

# Stage3 最終排序
CATEGORY_ORDER = {
  "建築物外觀": 1,
  "戶外環境": 2,
  "室內公共設施": 3,
  "客房": 4,
  "餐飲": 5
}

# Stage3 圖片填充策略
SELECTION_PATTERN = [
  "客房","客房","建築物外觀","室內公共設施","餐飲","戶外環境",
  "客房","室內公共設施","餐飲","戶外環境",
  "客房","室內公共設施","餐飲","戶外環境",
  "客房","室內公共設施","戶外環境",
  "客房","戶外環境","客房"
]

STAGE1_SYSTEM_PROMPT = """
你是專業住宿網站的圖片編輯。

任務:
1.過濾低品質或不適合展示的圖片。
2.將保留圖片分類為：
建築物外觀 / 戶外環境 / 室內公共設施 / 客房 / 餐飲

分類優先順序:
title -> 圖片內容 -> groupId

只可從提供的 image_id 中選擇。
不可新增或修改 image_id。

輸出Json:
{
  "kept": [
    {
      "image_id": "...",
      "category": "..."
    }
  ]
}
僅回傳 JSON。
"""

STAGE2_SYSTEM_PROMPT = """
你是專業住宿網站的圖片編輯。

任務:
從此類別的圖片中，
選出最具代表性、構圖佳、空間感佳的照片。
若畫面高度相似，僅保留最佳一張。

只可從提供的 image_id 中選擇。
不可新增或修改 image_id。

輸出 JSON:
{
  "selected": [
    {"image_id": "..."}
  ]
}
僅回傳 JSON。
"""


# ==========================================================
# UTILS
# 工具函式
# ==========================================================

async def simple_retry(func, *args, retries=3, sleep_sec=12, **kwargs):
  # """
  # 遇到 RateLimitError 時進行自動重試
  # """

  for i in range(retries):
    try:
      return await func(*args, **kwargs)

    except RateLimitError:
      print(f"Rate limit，等待 {sleep_sec} 秒後重試 ({i+1}/{retries})")
      await asyncio.sleep(sleep_sec)

  raise Exception("重試次數已達上限")


# ========================================
# Pipeline Class
# ========================================

class HotelImagePipeline:

  def __init__(self, hotel_id, client):
    self.hotel_id = hotel_id
    self.client = client

    # 控制單一 hotel OpenAI 併發
    self.sem = asyncio.Semaphore(CONCURRENCY)

    self.images = []
    self.stage1_result = []
    self.stage2_result = []
    self.stage3_result = []


  # ---------------------------------------------------
  # Stage0 : 抓取圖片資訊
  # ---------------------------------------------------

  def fetch_images(self):
    # """
    # 從 Agoda API 抓取圖片資訊, 包含ID、網址、標題、groudID等
    # """

    url = f"https://www.agoda.com/api/cronos/property/BelowFoldParams/GetSecondaryData?hotel_id={self.hotel_id}"
    res = requests.get(url, timeout=10)
    res.raise_for_status()

    data = res.json()

    image_list = data.get("mosaicInitData", {}).get("images", [])

    images = []
    seen = set()

    for obj in image_list:

      # 過濾 groupId 為空或 nearby
      group_id = obj.get("groupId")
      if not group_id or group_id == "nearby":
        continue

      # 取得 image url
      location = obj.get("location")

      if not location:
        continue

      full_url = "https:" + location if location.startswith("//") else location

      image_id = obj.get("id")

      # 使用 image_id 去重
      if image_id in seen:
        continue

      seen.add(image_id)

      images.append({
        "image_id": image_id,
        "title": obj.get("title"),
        "group": obj.get("group"),
        "groupId": obj.get("groupId"),
        "url": full_url,
      })

    self.images = images
    print(f"[{self.hotel_id}] 抓到 {len(images)} 張圖片")

    self._export_raw_images_csv()


  def _export_raw_images_csv(self):
    # """
    # 輸出原始 Agoda 圖片 CSV
    # """

    if not self.images:
      print(f"[{self.hotel_id}] 沒有圖片可輸出")
      return

    filename = f"{self.hotel_id}_agoda_images.csv"

    keys = self.images[0].keys()

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
      writer = csv.DictWriter(f, fieldnames=keys)
      writer.writeheader()
      writer.writerows(self.images)

    print(f"[{self.hotel_id}] 已輸出 {filename}")
    files.download(filename)


  # ------------------------------------
  # Stage 1: 圖片品質篩選、分類
  # ------------------------------------

  async def _stage1_batch(self, batch):
    # """
    # 單一 batch 的 Vision API 呼叫, 使用 semaphore 控制併發數。
    # """

    async with self.sem:

      content = []
      for img in batch:
        content.append({
          "type": "text",
          "text": f"image_id: {img["image_id"]}"
        })

        content.append({
            "type": "image_url",
            "image_url": {"url": img["url"]}
        })

      response = await simple_retry(
        self.client.chat.completions.create,
        model=STAGE1_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": STAGE1_SYSTEM_PROMPT},
            {"role": "user", "content": content}
        ],
        response_format={"type": "json_object"},
        timeout=60
      )

      result_json = json.loads(response.choices[0].message.content)

      batch_results = []

      for item in result_json.get("kept", []):
        batch_results.append({
          "image_id": item.get("image_id"),
          "category": item.get("category"),
        })

      return batch_results


  async def stage1(self):
    # """
    # Stage1 主流程：
    # 1. 進行非同步批次 OpenAI Vision API 呼叫。
    # 2. 將 AI 回傳的結果與原始資料進行合併。
    # 3. 透過規則執行類別校正。
    # """"

    print(f"[{self.hotel_id}] Stage1 開始")

    # --- 1. 分批 Vision 呼叫 ---
    tasks = []

    # 建立所有非同步任務
    for i in range(0, len(self.images), BATCH_SIZE):
      batch = self.images[i:i+BATCH_SIZE]
      tasks.append(self._stage1_batch(batch))

    # 同時執行所有任務
    results = await asyncio.gather(*tasks)

    # 收集 AI 結果
    stage1_kept = []
    for result in results:
      stage1_kept.extend(result)

    # --- 2. 合併原始欄位資料 ---

    #建立索引字典
    image_map = {img["image_id"]: img for img in self.images}

    merged = []

    for item in stage1_kept:
      original = image_map.get(item["image_id"], {})

      merged.append({
        "image_id": item["image_id"],
        "url": original.get("url"),
        "category": item["category"],
        "title": original.get("title"),
        "groupId": original.get("groupId")
      })


    # --- 3. 類別校正 ---
    def normalize_category(item):

      title = (item.get("title") or "").lower()
      groupId = (item.get("groupId") or "").lower()
      ai_category = item.get("category")

      if groupId == "room":
        return "客房"

      if groupId == "dining":
        return "餐飲"

      if "exterior" in title:
        return "建築物外觀"

      if "restaurant" in title:
        return "餐飲"

      if "lobby" in title:
        return "室內公共設施"

      return ai_category


    corrected = []

    for item in merged:
      item["category"] = normalize_category(item)
      corrected.append(item)

    self.stage1_result = corrected

    print(f"[{self.hotel_id}] Stage1 保留 {len(corrected)} 張圖片")

    self._export_stage1_json()

    return corrected


  def _export_stage1_json(self):
    # """
    # 輸出 Stage1 JSON
    # """

    filename = f"{self.hotel_id}_stage1_result.json"

    with open(filename, "w", encoding="utf-8") as f:
      json.dump(self.stage1_result, f, ensure_ascii=False, indent=2)

    print(f"[{self.hotel_id}] 已輸出 {filename}")
    files.download(filename)


  # ------------------------------------
  # Stage 2: 各類別內圖片做精選
  # ------------------------------------

  async def _stage2_category(self, category, items):
    # """
    # 單一類別內做 Vision 精選, 使用 semaphore 控制併發數。
    # """

    limit = CATEGORY_LIMIT.get(category, 0)

    print(f"[{self.hotel_id}] 處理類別: {category} (共 {len(items)} 張，上限 {limit})")

    # 圖片過多則抽樣
    if len(items) > MAX_VISION_INPUT:
      print(f"[{self.hotel_id}] {category}類別超過 {MAX_VISION_INPUT} 張，隨機取 {MAX_VISION_INPUT} 張")
      items = random.sample(items, MAX_VISION_INPUT)
      items = items[:MAX_VISION_INPUT]

    # 若數量 <= limit，直接保留
    if len(items) <= limit:
      return [
        {
          "image_id": i.get("image_id"),
          "url": i.get("url"),
          "category": category
        }
        for i in items
      ]

    async with self.sem:
      content = [{
        "type": "text",
        "text": f"\n\n本類別最多保留：{limit} 張"
      }]

      for img in items:
        content.append({
          "type": "text",
          "text": f"image_id: {img["image_id"]}"
        })
        content.append({
          "type": "image_url",
          "image_url": {"url": img["url"]}
        })

      response = await simple_retry(
        self.client.chat.completions.create,
        model=STAGE2_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": STAGE2_SYSTEM_PROMPT},
            {"role": "user", "content": content}
        ],
        response_format={"type": "json_object"},
        timeout=60
      )

      result_json = json.loads(response.choices[0].message.content)

      selected = result_json.get("selected", [])[:limit]

      #建立索引字典
      image_map = {img["image_id"]: img for img in items}

      return [
        {
          "image_id": s.get("image_id"),
          "url": image_map.get(s.get("image_id"), {}).get("url"),
          "category": category
        }
        for s in selected
      ]


  async def stage2(self):
    # """
    # Stage2 主流程：進行非同步批次 OpenAI Vision API 呼叫。
    # """

    print(f"[{self.hotel_id}] Stage2 開始")

    grouped = defaultdict(list)

    for item in self.stage1_result:
      grouped[item["category"]].append(item)

    tasks = []

    # 建立所有非同步任務
    for category, items in grouped.items():
      tasks.append(self._stage2_category(category, items))

    # 同時執行所有任務
    results = await asyncio.gather(*tasks)

    # 收集 AI 結果
    final_results = []
    for result in results:
      final_results.extend(result)

    self.stage2_result = final_results

    print(f"[{self.hotel_id}] Stage2 保留 {len(final_results)} 張圖片")

    self._export_stage2_html()

    return final_results


  def _export_stage2_html(self):
    # """
    # Stage2 HTML 輸出
    # """

    filename = f"{self.hotel_id}_stage2_result.html"

    css = """
    <style>
    img {
        width: 200px;
        height: 140px;
        margin: 6px;
    }
    </style>
    """

    html_content = [css]
    html_content.extend([
      f"<img src='{img["url"]}' title='{img["category"]}'>"
      for img in self.stage2_result
    ])

    with open(filename, "w", encoding="utf-8-sig") as f:
      f.write("\n".join(html_content))

    print(f"[{self.hotel_id}] 已輸出 {filename}")
    files.download(filename)


  # ========================================
  # Stage 3: 平衡各類別選出 10 張圖片
  # ========================================

  def stage3(self):
    # """
    # Stage 3主流程:
    # 圖片>10張 -> 依據填充策略進行挑選，然後排序
    # """

    print(f"[{self.hotel_id}] Stage3 開始")

    stage2_result = self.stage2_result

    # 圖片10張以下
    if len(stage2_result) <= 10:
      final_sorted = sorted(
        stage2_result,
        key=lambda x: CATEGORY_ORDER.get(x["category"], 99)
      )

    else:
      # 分組
      grouped = defaultdict(list)
      for item in stage2_result:
        grouped[item["category"]].append(item)

      final = []
      used_ids = set()

      # 依照 pattern 順序填充
      for category in SELECTION_PATTERN:
        if len(final) >= 10:
          break

        if category in grouped and len(grouped[category]) > 0:
          while grouped[category]:

            # 取第一張尚未使用的
            candidate = grouped[category].pop(0)

            if candidate["image_id"] not in used_ids:
              final.append(candidate)
              used_ids.add(candidate["image_id"])
              break

      # 排序
      final_sorted = sorted(
        final,
        key=lambda x: CATEGORY_ORDER.get(x["category"], 99)
      )

    self.stage3_result = final_sorted

    print(f"[{self.hotel_id}] Stage3 挑選 {len(final_sorted)} 張圖片")

    self._export_stage3_html()
    self._export_stage3_txt()


  def _export_stage3_html(self):
    # """
    # Stage3 HTML 輸出
    # """

    html_filename = f"{self.hotel_id}_stage3_result.html"

    css = """
    <style>
    img {
        width: 250px;
        height: 175px;
        margin: 6px;
    }
    </style>
    """

    html_content = [css]

    html_content.append(f"<a href='{self.hotel_id}_stage1_result.json' target='_blank'>Stage1</a>\t")
    html_content.append(f"<a href='{self.hotel_id}_stage2_result.html' target='_blank'>Stage2</a>\t")
    html_content.append(f"<a href='{self.hotel_id}_stage3_result.txt' target='_blank'>Stage3_Result</a><br>")

    html_content.extend(f"<img src='{item["url"]}'>" for item in self.stage3_result if item.get("url"))

    html_content.append("<hr>")

    with open(html_filename, "w", encoding="utf-8-sig") as f:
      f.write("\n".join(html_content))

    print(f"[{self.hotel_id}] 已輸出 {html_filename}")
    files.download(html_filename)


  def _export_stage3_txt(self):
    # """
    # Stage3 TXT 輸出
    # """

    txt_filename = f"{self.hotel_id}_stage3_result.txt"

    txt_content = [item["url"] for item in self.stage3_result if item.get("url")]

    with open(txt_filename, "w", encoding="utf-8-sig") as f:
        f.write("\n".join(txt_content))

    print(f"[{self.hotel_id}] 已輸出 {txt_filename}")
    files.download(txt_filename)


  # ------------------------------------
  # Run
  # ------------------------------------

  async def run(self):

    self.fetch_images()
    await self.stage1()
    await self.stage2()
    self.stage3()


# ========================================
# Hotels 依序執行
# ========================================

async def run_hotels_sequential():

  client = AsyncOpenAI(api_key=OPENAI_API_KEY, timeout=60)

  for hotel_id in HOTEL_LIST:
    print(f"\n========== 開始處理 Hotel {hotel_id} ==========")

    pipeline = HotelImagePipeline(hotel_id, client)

    await pipeline.run()

    print(f"========== Hotel {hotel_id} 完成 ==========\n")


# ========================================
# Execute
# 執行
# ========================================

await run_hotels_sequential()

print("全部處理完成")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

========== 開始處理 Hotel 545854 ==========
[545854] 抓到 89 張圖片
[545854] 已輸出 545854_agoda_images.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[545854] Stage1 開始
[545854] Stage1 保留 87 張圖片
[545854] 已輸出 545854_stage1_result.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

[32198776] 已輸出 32198776_stage3_result.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

========== Hotel 32198776 完成 ==========

全部處理完成
